# Live Change: watch mirroring replicate in real time

The proof moment: change a row **in the source Azure SQL database**, then watch
the change arrive in Fabric's replicated copy, with no pipeline, no refresh, and
no code in between.

What changes is driven by the sector's `mirroring.json` spec (which table, which
row, which column, and how).

In [ ]:
import base64, json, time
import notebookutils

# The per-sector spec is injected by the deployer as base64 of demos/<demo>/mirroring.json
SPEC = json.loads(base64.b64decode("{{MIRRORING_SPEC_B64}}").decode("utf-8"))

def _sql_access_token():
    errors = []
    for audience in ("https://database.windows.net/", "https://database.windows.net"):
        try:
            tok = notebookutils.credentials.getToken(audience)
            if tok:
                return tok
        except Exception as e:  # noqa: BLE001
            errors.append(f"{audience}: {e}")
    raise RuntimeError("Could not acquire an Azure SQL access token via "
                       "notebookutils.credentials.getToken: " + "; ".join(errors))

SQL_SERVER = "{{SQL_SERVER}}"
SQL_DATABASE = "{{SQL_DATABASE}}"
WORKSPACE_ID = "{{WORKSPACE_ID}}"
MIRRORED_DB_ID = "{{MIRRORED_DB_ID}}"
LC = SPEC["liveChange"]

ACCESS_TOKEN = _sql_access_token()

def mirrored(table):
    path = (
        f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
        f"{MIRRORED_DB_ID}/Tables/dbo/{table}"
    )
    return spark.read.format("delta").load(path)

def run_sql(statement):
    """Execute a statement against the SOURCE Azure SQL database (Entra auth)."""
    jvm = spark._sc._jvm
    props = jvm.java.util.Properties()
    props.setProperty("accessToken", ACCESS_TOKEN)
    jvm.java.lang.Class.forName("com.microsoft.sqlserver.jdbc.SQLServerDriver")
    url = (f"jdbc:sqlserver://{SQL_SERVER}:1433;database={SQL_DATABASE};"
           "encrypt=true;trustServerCertificate=false;loginTimeout=60;")
    conn = jvm.java.sql.DriverManager.getConnection(url, props)
    stmt = conn.createStatement()
    n = stmt.executeUpdate(statement)
    stmt.close(); conn.close()
    return n

print(LC.get("story", ""))
print("Ready")

In [ ]:
# Pick a real row from the mirrored copy and decide the new value (per spec)
table = LC["table"]; key = LC["keyColumn"]; mcol = LC["mutateColumn"]; kind = LC["mutateKind"]
disp = LC.get("displayColumn")
mdf = mirrored(table)

def _as_num(v):
    if isinstance(v, bool):
        return 1.0 if v else 0.0
    return float(v)

if kind == "number":
    cand = mdf.filter(f"`{mcol}` is not null").orderBy(key).limit(1).collect()
    if not cand:
        raise RuntimeError(f"No row in {table} with a value in {mcol} to change")
    row = cand[0]; TARGET_KEY = row[key]; old_val = float(row[mcol])
    NEW_VALUE = round(old_val * float(LC.get("numberFactor", 1.15)) + 1, 2)
    set_expr = str(NEW_VALUE); expected = NEW_VALUE
elif kind == "flag":
    flag_to = LC.get("flagTo", 1)
    cand = mdf.filter(f"`{mcol}` <> {flag_to} or `{mcol}` is null").orderBy(key).limit(1).collect()
    if not cand:
        raise RuntimeError(f"No row in {table} where {mcol} differs from {flag_to}")
    row = cand[0]; TARGET_KEY = row[key]; old_val = row[mcol]
    NEW_VALUE = flag_to; set_expr = str(flag_to); expected = flag_to
elif kind == "status":
    status_to = LC["statusTo"]
    cand = mdf.filter(f"`{mcol}` <> '{status_to}' or `{mcol}` is null").orderBy(key).limit(1).collect()
    if not cand:
        raise RuntimeError(f"No row in {table} where {mcol} differs from {status_to}")
    row = cand[0]; TARGET_KEY = row[key]; old_val = row[mcol]
    NEW_VALUE = status_to; set_expr = f"'{status_to}'"; expected = status_to
else:
    raise RuntimeError(f"Unknown mutateKind: {kind}")

label = row[disp] if disp else TARGET_KEY
print(f"Target row: {table}.{key} = {TARGET_KEY}" + (f"  ({disp} = {label})" if disp else ""))
print(f"Current {mcol} in the MIRRORED copy: {old_val}")
print(f"Will set {mcol} -> {NEW_VALUE} in the SOURCE Azure SQL database")

In [ ]:
# Make the change IN THE SOURCE database (Azure SQL), not in Fabric
key_literal = f"'{TARGET_KEY}'" if isinstance(TARGET_KEY, str) else str(TARGET_KEY)
n = run_sql(f"UPDATE dbo.{table} SET {mcol} = {set_expr} WHERE {key} = {key_literal}")
print(f"UPDATE executed against Azure SQL ({n} row affected)")

still = mirrored(table).filter(f"{key} = {key_literal}").collect()[0][mcol]
print(f"  mirrored copy still shows: {still}   (not replicated yet)")

In [ ]:
# Watch Fabric catch up BY ITSELF (no pipeline, no refresh)
POLL_EVERY = 10      # seconds
MAX_WAIT = 600

numeric_kind = kind in ("number", "flag")
start = time.time()
replicated = None
while time.time() - start < MAX_WAIT:
    cur = mirrored(table).filter(f"{key} = {key_literal}").collect()[0][mcol]
    elapsed = int(time.time() - start)
    if numeric_kind:
        matched = abs(_as_num(cur) - _as_num(expected)) < 0.5
    else:
        matched = str(cur) == str(expected)
    if matched:
        replicated = elapsed
        print(f"[{elapsed:>3}s] mirrored copy: {cur}   REPLICATED in ~{elapsed}s")
        break
    print(f"[{elapsed:>3}s] mirrored copy: {cur}   (waiting for replication...)")
    time.sleep(POLL_EVERY)

if replicated is None:
    print(f"Change not visible after {MAX_WAIT}s. The first sync after seeding can lag")
    print("a few minutes. Check the mirrored database replication status, then re-run.")
else:
    print("\nNothing moved that data except Fabric Mirroring itself:")
    print("  no pipeline, no schedule, no refresh, no code.")